# AE3 - Transformación y Enriquecimiento de Datos
**Proyecto integrador III - Ventas de videojuegos (Hugging Face: 0tizm0/video-games)**

**Necesidad:** orientar la decisión de qué géneros, plataformas y editores
conviene priorizar en un catálogo de lanzamientos, a partir del histórico de ventas.

| # | Pregunta de negocio | Variables | Resultado esperado |
|---|---|---|---|
| P1 | ¿Qué géneros aportan más ventas globales? | Genre, Global_Sales | Tabla por género con ventas y participación % |
| P2 | ¿Qué plataformas y fabricantes concentran las ventas y cómo se reparten por región? | Platform, Fabricante, *_Sales | Tabla por plataforma y por fabricante |
| P3 | ¿Cómo evolucionaron las ventas por año? | Year_of_Release, Global_Sales | Serie anual con variación interanual |
| P4 | ¿Qué editores lideran ventas y éxitos (Is_Hit)? | Publisher, Global_Sales, Is_Hit | Top 15 editores |
| P5 | ¿Los juegos mejor calificados por la crítica venden más? | Critic_Score, Global_Sales | Tabla por rango de calificación |

**Limitaciones declaradas:** las ventas están en millones de unidades (no en dinero);
no hay costos, por lo que no se puede hablar de rentabilidad; Critic_Score y
User_Score tienen muchos nulos; el análisis es descriptivo (no causal).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DIR_DATOS = Path("datos")
DIR_TABLAS = Path("tablas")
DIR_REPORTES = Path("reportes")
for d in (DIR_DATOS, DIR_TABLAS, DIR_REPORTES):
    d.mkdir(parents=True, exist_ok=True)

## 1. Carga y copia sin modificar

In [2]:
def cargar_dataset() -> pd.DataFrame:
    """Descarga el dataset de Hugging Face; si falla, usa la copia local."""
    local = DIR_DATOS / "video-games_original.csv"
    alterno = DIR_DATOS / "video-games.csv"      # nombre que descarga HFDatasetLoader
    try:
        from datasets import load_dataset
        df = load_dataset("0tizm0/video-games", split="train").to_pandas()
        df.to_csv(local, index=False)
    except Exception as e:
        fuente = local if local.exists() else alterno
        if not fuente.exists():
            raise
        print(f"Sin descarga ({type(e).__name__}); se usa la copia local {fuente}.")
        df = pd.read_csv(fuente)
    return df

df_original = cargar_dataset()   # versión sin modificar, solo para comparar
df = df_original.copy()
print(df_original.shape)
df_original.head()

C:\Users\jonat\Documents\Machine-Learning-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\jonat\Documents\Machine-Learning-\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jonat\.cache\huggingface\hub\datasets--0tizm0--video-games. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/16713 [00:00<?, ? examples/s]

Generating train split:  60%|█████▉    | 10000/16713 [00:00<00:00, 63694.82 examples/s]

Generating train split: 100%|██████████| 16713/16713 [00:00<00:00, 61361.92 examples/s]

Generating train split: 100%|██████████| 16713/16713 [00:00<00:00, 60338.19 examples/s]

(16713, 18)


,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating,Year_missing,Is_Hit
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0,Nintendo,E,0.0,1.0
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,Unknown,None,0.0,1.0
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E,0.0,1.0
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0,Nintendo,E,0.0,1.0
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,Unknown,None,0.0,1.0


## 2. Transformación y validación

In [3]:
def resumen(d: pd.DataFrame) -> dict:
    return {"filas": len(d), "columnas": d.shape[1],
            "nulos_totales": int(d.isna().sum().sum()),
            "duplicados_exactos": int(d.duplicated().sum())}

antes = resumen(df)
print("Nulos por columna (antes):")
print(df.isna().sum()[lambda s: s > 0])

Nulos por columna (antes):
Year_of_Release     269
Critic_Score       8576
Critic_Count       8576
User_Score         9123
User_Count         9123
Rating             6764
dtype: int64


In [4]:
# 2.1 Limpieza de texto y nombres faltantes
cols_texto = ["Name", "Platform", "Genre", "Publisher", "Developer", "Rating"]
for c in cols_texto:
    df[c] = df[c].astype("string").str.strip()

sin_nombre = int(df["Name"].isna().sum())
df = df.dropna(subset=["Name"])           # sin nombre no se puede identificar el juego
print(f"Registros sin nombre eliminados: {sin_nombre}")

# 2.2 Duplicados: exactos se eliminan; repetidos por (Name, Platform, Year) solo se reportan
dup_exactos = int(df.duplicated().sum())
df = df.drop_duplicates()
dup_clave = int(df.duplicated(subset=["Name", "Platform", "Year_of_Release"]).sum())
print(f"Duplicados exactos eliminados: {dup_exactos}")
print(f"Repetidos por Name+Platform+Year (se conservan, tienen cifras distintas): {dup_clave}")

Registros sin nombre eliminados: 0


Duplicados exactos eliminados: 0
Repetidos por Name+Platform+Year (se conservan, tienen cifras distintas): 1


In [5]:
# 2.3 Tipos: años y conteos como enteros con nulos permitidos
for c in ["Year_of_Release", "Critic_Count", "User_Count", "Year_missing", "Is_Hit"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").round().astype("Int64")

# 2.4 Categorías: valores desconocidos explícitos, sin imputar puntajes
df["Developer"] = df["Developer"].replace({"Unknown": pd.NA}).fillna("Sin dato")
df["Publisher"] = df["Publisher"].fillna("Sin dato")
df["Rating"] = df["Rating"].fillna("Sin clasificar")
df["Genre"] = df["Genre"].fillna("Sin dato")

mapa_rating = {"E": "Todos", "E10+": "Todos +10", "T": "Adolescentes", "M": "Maduro 17+",
               "AO": "Solo adultos", "K-A": "Niños a adultos", "EC": "Primera infancia",
               "RP": "Pendiente de clasificar"}
df["Rating_Desc"] = df["Rating"].map(mapa_rating).fillna(df["Rating"])

# 2.5 Validaciones numéricas
regiones = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"]
negativos = int((df[regiones + ["Global_Sales"]] < 0).sum().sum())
dif = (df[regiones].sum(axis=1) - df["Global_Sales"]).abs()
incoherentes = int((dif > 0.05).sum())
fuera_rango_critica = int(((df["Critic_Score"] < 0) | (df["Critic_Score"] > 100)).sum())
fuera_rango_user = int(((df["User_Score"] < 0) | (df["User_Score"] > 10)).sum())
print("Ventas negativas:", negativos)
print("Filas donde suma regional difiere de Global_Sales en más de 0.05M:", incoherentes)
print("Critic_Score fuera de 0-100:", fuera_rango_critica, "| User_Score fuera de 0-10:", fuera_rango_user)

Ventas negativas: 0
Filas donde suma regional difiere de Global_Sales en más de 0.05M: 0
Critic_Score fuera de 0-100: 0 | User_Score fuera de 0-10: 0


In [6]:
# 2.6 Comparación antes / después
despues = resumen(df)
comparacion = pd.DataFrame({"antes": antes, "despues": despues})
comparacion["cambio"] = comparacion["despues"] - comparacion["antes"]
comparacion.to_csv(DIR_TABLAS / "00_comparacion_antes_despues.csv")
comparacion

,antes,despues,cambio
filas,16713,16713,0
columnas,18,19,1
nulos_totales,42431,35667,-6764
duplicados_exactos,0,0,0


In [7]:
tipos = pd.DataFrame({"tipo_antes": df_original.dtypes.astype(str)})
tipos["tipo_despues"] = df.dtypes.astype(str)
tipos["nulos_antes"] = df_original.isna().sum()
tipos["nulos_despues"] = df.isna().sum()
tipos

,tipo_antes,tipo_despues,nulos_antes,nulos_despues
Name,object,string,0,0
Platform,object,string,0,0
Year_of_Release,float64,Int64,269,269
Genre,object,string,0,0
Publisher,object,string,0,0
NA_Sales,float64,float64,0,0
EU_Sales,float64,float64,0,0
JP_Sales,float64,float64,0,0
Other_Sales,float64,float64,0,0
Global_Sales,float64,float64,0,0


## 3. Enriquecimiento (solo con variables existentes)
| Variable nueva | Regla | Utilidad |
|---|---|---|
| Fabricante | Mapa Platform → Nintendo/Sony/Microsoft/Sega/PC/Otros | P2 |
| Decada | (Año // 10) * 10 | P3 |
| Region_Dominante | Región con mayor venta del juego | P2 |
| Rango_Ventas | Cortes de Global_Sales (millones) | P1, P4 |
| Rango_Critica | Cortes de Critic_Score; "Sin puntaje" si es nulo | P5 |

In [8]:
fabricante = {
    **{p: "Nintendo" for p in ["3DS", "DS", "GB", "GBA", "GC", "N64", "NES", "SNES", "Wii", "WiiU"]},
    **{p: "Sony" for p in ["PS", "PS2", "PS3", "PS4", "PSP", "PSV"]},
    **{p: "Microsoft" for p in ["X360", "XB", "XOne"]},
    **{p: "Sega" for p in ["DC", "GEN", "GG", "SAT", "SCD"]},
    "PC": "PC",
}
df["Fabricante"] = df["Platform"].map(fabricante).fillna("Otros")
df["Decada"] = (df["Year_of_Release"] // 10 * 10).astype("Int64").astype("string") + "s"
df["Region_Dominante"] = (df[regiones].idxmax(axis=1)
                          .str.replace("_Sales", "", regex=False)
                          .replace({"NA": "Norteamérica", "EU": "Europa", "JP": "Japón", "Other": "Otras"}))
df["Rango_Ventas"] = pd.cut(df["Global_Sales"], [-0.001, 0.1, 0.5, 1, 5, np.inf],
                            labels=["<0.1M", "0.1-0.5M", "0.5-1M", "1-5M", ">5M"]).astype("string")
df["Rango_Critica"] = pd.cut(df["Critic_Score"], [0, 60, 75, 90, 100],
                             labels=["<=60", "61-75", "76-90", "91-100"]).astype("string").fillna("Sin puntaje")
df[["Name", "Platform", "Fabricante", "Decada", "Region_Dominante", "Rango_Ventas", "Rango_Critica"]].head()

,Name,Platform,Fabricante,Decada,Region_Dominante,Rango_Ventas,Rango_Critica
0,Wii Sports,Wii,Nintendo,2000s,Norteamérica,>5M,76-90
1,Super Mario Bros.,NES,Nintendo,1980s,Norteamérica,>5M,Sin puntaje
2,Mario Kart Wii,Wii,Nintendo,2000s,Norteamérica,>5M,76-90
3,Wii Sports Resort,Wii,Nintendo,2000s,Norteamérica,>5M,76-90
4,Pokemon Red/Pokemon Blue,GB,Nintendo,1990s,Norteamérica,>5M,Sin puntaje


## 4. Respuestas a las preguntas de negocio
Unidades: ventas en **millones de unidades**.

In [9]:
total_global = df["Global_Sales"].sum()

def exportar(t: pd.DataFrame, nombre: str) -> pd.DataFrame:
    t.to_csv(DIR_TABLAS / nombre, index=False)
    return t

# P1 - Género
p1 = (df.groupby("Genre").agg(Juegos=("Name", "count"), Ventas_Globales_M=("Global_Sales", "sum"))
        .assign(Participacion_Pct=lambda t: t.Ventas_Globales_M / total_global * 100,
                Ventas_Promedio_por_Juego_M=lambda t: t.Ventas_Globales_M / t.Juegos)
        .sort_values("Ventas_Globales_M", ascending=False).round(2).reset_index())
exportar(p1, "P1_ventas_por_genero.csv")

,Genre,Juegos,Ventas_Globales_M,Participacion_Pct,Ventas_Promedio_por_Juego_M
0,Action,3369,1745.26,19.57,0.52
1,Sports,2348,1332.00,14.94,0.57
2,Shooter,1323,1052.94,11.81,0.80
3,Role-Playing,1498,934.35,10.48,0.62
4,Platform,888,828.08,9.29,0.93
5,Misc,1750,803.18,9.01,0.46
6,Racing,1249,728.90,8.17,0.58
7,Fighting,849,447.48,5.02,0.53
8,Simulation,873,390.13,4.37,0.45
9,Puzzle,580,243.02,2.73,0.42


In [10]:
# P2 - Plataforma y fabricante
agg_reg = {"Juegos": ("Name", "count"), "NA_M": ("NA_Sales", "sum"), "EU_M": ("EU_Sales", "sum"),
           "JP_M": ("JP_Sales", "sum"), "Otras_M": ("Other_Sales", "sum"), "Global_M": ("Global_Sales", "sum")}
p2a = (df.groupby(["Fabricante", "Platform"]).agg(**agg_reg)
         .assign(Participacion_Pct=lambda t: t.Global_M / total_global * 100)
         .sort_values("Global_M", ascending=False).round(2).reset_index())
p2b = (df.groupby("Fabricante").agg(**agg_reg)
         .assign(Participacion_Pct=lambda t: t.Global_M / total_global * 100)
         .sort_values("Global_M", ascending=False).round(2).reset_index())
exportar(p2a, "P2a_ventas_por_plataforma.csv")
exportar(p2b, "P2b_ventas_por_fabricante.csv")

,Fabricante,Juegos,NA_M,EU_M,JP_M,Otras_M,Global_M,Participacion_Pct
0,Sony,6720,1544.34,1104.08,473.86,466.26,3588.34,40.24
1,Nintendo,6270,1762.40,780.74,762.03,191.66,3499.50,39.24
2,Microsoft,2333,882.28,383.30,14.15,108.75,1389.33,15.58
3,PC,974,94.53,142.44,0.17,22.38,260.30,2.92
4,Otros,157,90.60,5.47,3.15,0.91,100.23,1.12
5,Sega,259,26.42,8.11,43.98,1.28,79.83,0.90


In [11]:
# P3 - Evolución anual
p3 = (df.dropna(subset=["Year_of_Release"]).groupby("Year_of_Release")
        .agg(Juegos=("Name", "count"), Ventas_Globales_M=("Global_Sales", "sum"))
        .sort_index())
p3["Variacion_Interanual_Pct"] = p3["Ventas_Globales_M"].pct_change() * 100
p3 = p3.round(2).reset_index().rename(columns={"Year_of_Release": "Anio"})
exportar(p3, "P3_ventas_por_anio.csv")

,Anio,Juegos,Ventas_Globales_M,Variacion_Interanual_Pct
0,1980,9,11.38,NaN
1,1981,46,35.77,214.32
2,1982,36,28.86,-19.32
3,1983,17,16.79,-41.82
4,1984,14,50.36,199.94
5,1985,14,53.94,7.11
6,1986,21,37.07,-31.28
7,1987,16,21.74,-41.35
8,1988,15,47.22,117.20
9,1989,17,73.45,55.55


In [12]:
# P4 - Top 15 editores
p4 = (df.groupby("Publisher").agg(Juegos=("Name", "count"), Ventas_Globales_M=("Global_Sales", "sum"),
                                  Exitos_Is_Hit=("Is_Hit", "sum"))
        .assign(Participacion_Pct=lambda t: t.Ventas_Globales_M / total_global * 100,
                Tasa_Exito_Pct=lambda t: t.Exitos_Is_Hit / t.Juegos * 100)
        .sort_values("Ventas_Globales_M", ascending=False).head(15).round(2).reset_index())
exportar(p4, "P4_top15_editores.csv")

,Publisher,Juegos,Ventas_Globales_M,Exitos_Is_Hit,Participacion_Pct,Tasa_Exito_Pct
0,Nintendo,706,1788.81,459,20.06,65.01
1,Electronic Arts,1356,1116.96,688,12.53,50.74
2,Activision,985,731.16,343,8.20,34.82
3,Sony Computer Entertainment,687,606.48,279,6.80,40.61
4,Ubisoft,932,471.32,254,5.29,27.25
5,Take-Two Interactive,422,403.82,162,4.53,38.39
6,THQ,715,338.44,217,3.80,30.35
7,Konami Digital Entertainment,834,282.39,139,3.17,16.67
8,Sega,636,270.30,153,3.03,24.06
9,Namco Bandai Games,939,254.62,138,2.86,14.7


In [13]:
# P5 - Crítica vs ventas (solo juegos con puntaje)
orden = ["<=60", "61-75", "76-90", "91-100"]
con_critica = df[df["Rango_Critica"] != "Sin puntaje"]
p5 = (con_critica.groupby("Rango_Critica")
        .agg(Juegos=("Name", "count"), Ventas_Globales_M=("Global_Sales", "sum"),
             Venta_Promedio_M=("Global_Sales", "mean"), Venta_Mediana_M=("Global_Sales", "median"))
        .reindex(orden).round(2).reset_index())
rho = con_critica["Critic_Score"].astype(float).corr(con_critica["Global_Sales"], method="spearman")
print(f"Correlación de Spearman Critic_Score vs Global_Sales: {rho:.2f} "
      f"(n={len(con_critica)}; {len(df) - len(con_critica)} juegos sin puntaje excluidos)")
exportar(p5, "P5_ventas_por_rango_critica.csv")

Correlación de Spearman Critic_Score vs Global_Sales: 0.39 (n=8137; 8576 juegos sin puntaje excluidos)


,Rango_Critica,Juegos,Ventas_Globales_M,Venta_Promedio_M,Venta_Mediana_M
0,<=60,2069,561.01,0.27,0.14
1,61-75,3138,1377.38,0.44,0.20
2,76-90,2673,2838.56,1.06,0.50
3,91-100,257,829.73,3.23,1.72


In [14]:
for nombre, t in {"P1": p1, "P2a": p2a, "P2b": p2b, "P3": p3, "P4": p4, "P5": p5}.items():
    print(f"\n=== {nombre} ===")
    print(t.head(10).to_string(index=False))


=== P1 ===
       Genre  Juegos  Ventas_Globales_M  Participacion_Pct  Ventas_Promedio_por_Juego_M
      Action    3369            1745.26              19.57                         0.52
      Sports    2348            1332.00              14.94                         0.57
     Shooter    1323            1052.94              11.81                         0.80
Role-Playing    1498             934.35              10.48                         0.62
    Platform     888             828.08               9.29                         0.93
        Misc    1750             803.18               9.01                         0.46
      Racing    1249             728.90               8.17                         0.58
    Fighting     849             447.48               5.02                         0.53
  Simulation     873             390.13               4.37                         0.45
      Puzzle     580             243.02               2.73                         0.42

=== P2a ===
Fabrica

### Verificaciones de consistencia

In [15]:
assert np.isclose(p1["Ventas_Globales_M"].sum(), total_global, atol=0.5), "P1 no cuadra"
assert np.isclose(p2a["Global_M"].sum(), total_global, atol=0.5), "P2a no cuadra"
assert np.isclose(p2b["Global_M"].sum(), total_global, atol=0.5), "P2b no cuadra"
assert abs(p1["Participacion_Pct"].sum() - 100) < 0.5, "Participación P1 no suma 100"
assert abs(p2b["Participacion_Pct"].sum() - 100) < 0.5, "Participación P2b no suma 100"
print("Totales y porcentajes consistentes. Total global (M):", round(total_global, 2))

Totales y porcentajes consistentes. Total global (M): 8917.53


## 5. Exportaciones: dataset preparado y resumen .txt (se regeneran en cada ejecución)

In [16]:
df.to_csv(DIR_DATOS / "video_games_preparado.csv", index=False)

def generar_resumen_txt(d: pd.DataFrame, ruta: Path) -> None:
    with open(ruta, "w", encoding="utf-8") as f:
        f.write("DESCRIPCIÓN DEL DATAFRAME TRANSFORMADO Y ENRIQUECIDO\n")
        f.write("=" * 60 + "\n")
        f.write(f"Filas: {d.shape[0]}\nColumnas: {d.shape[1]}\n\n")
        f.write("Columnas y tipos:\n" + d.dtypes.astype(str).to_string() + "\n\n")
        f.write("Valores nulos por columna:\n" + d.isna().sum().to_string() + "\n\n")
        f.write(f"Duplicados exactos: {int(d.duplicated().sum())}\n\n")
        f.write("Estadísticas descriptivas (numéricas):\n")
        f.write(d.describe().T.round(2).to_string() + "\n\n")
        f.write("Estadísticas descriptivas (categóricas):\n")
        f.write(d.select_dtypes(include=["string", "object"]).describe().T.to_string() + "\n")

generar_resumen_txt(df, DIR_REPORTES / "descripcion_df_final.txt")

# Verificar que lo exportado coincide con el DataFrame en memoria
chk = pd.read_csv(DIR_DATOS / "video_games_preparado.csv")
assert chk.shape == df.shape, "El CSV exportado no coincide con el DataFrame"
print("Exportado:", chk.shape, "| archivos en tablas/:", len(list(DIR_TABLAS.glob("*.csv"))))

Exportado: (16713, 24) | archivos en tablas/: 7


## 6. Interpretación de hallazgos
*(Complétala con los números reales después de ejecutar: una frase por pregunta
que diga qué muestra la tabla, para qué sirve a la necesidad y qué no permite concluir.)*